In [ ]:
04!pip install datasets==2.14.6

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Succe

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [ ]:
for i in range(20):
    print(f"--- Example {i} ---")
    print(dataset["train"][i])
    print()

--- Example 0 ---
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

--- Example 1 ---
{'id': '1', 'tokens': ['Peter', 'Blackburn'], 'pos_tags': [22, 22], 'chunk_tags': [11, 12], 'ner_tags': [1, 2]}

--- Example 2 ---
{'id': '2', 'tokens': ['BRUSSELS', '1996-08-22'], 'pos_tags': [22, 11], 'chunk_tags': [11, 12], 'ner_tags': [5, 0]}

--- Example 3 ---
{'id': '3', 'tokens': ['The', 'European', 'Commission', 'said', 'on', 'Thursday', 'it', 'disagreed', 'with', 'German', 'advice', 'to', 'consumers', 'to', 'shun', 'British', 'lamb', 'until', 'scientists', 'determine', 'whether', 'mad', 'cow', 'disease', 'can', 'be', 'transmitted', 'to', 'sheep', '.'], 'pos_tags': [12, 22, 22, 38, 15, 22, 28, 38, 15, 16, 21, 35, 24, 35, 37, 16, 21, 15, 24, 41, 15, 16, 21, 21, 20, 37, 40, 35, 21, 7], 'chunk_tags': [11, 12

## Random Retrieval k=5

In [ ]:
# ===============================
# Install + Imports
# ===============================
!pip install openai tqdm

from datasets import load_dataset
import random
import re
from tqdm import tqdm
import getpass
from openai import OpenAI

# ===============================
# API KEY (hidden input)
# ===============================
api_key = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

# ===============================
# Load dataset
# ===============================
dataset = load_dataset("conll2003")

# ===============================
# Tag mapping
# ===============================
tag_map = {
    0: "O",
    1: "B-PER", 2: "I-PER",
    3: "B-ORG", 4: "I-ORG",
    5: "B-LOC", 6: "I-LOC",
    7: "B-MISC", 8: "I-MISC"
}

entity_types = ["PER", "LOC", "ORG", "MISC"]

# ===============================
# Extract entities from ner_tags
# ===============================
def extract_entities(tokens, tags):
    entities = []
    current = []
    current_type = None

    for token, tag in zip(tokens, tags):
        tag_name = tag_map[tag]

        if tag_name.startswith("B-"):
            if current:
                entities.append((current_type, " ".join(current)))
            current = [token]
            current_type = tag_name[2:]

        elif tag_name.startswith("I-") and current:
            current.append(token)

        else:
            if current:
                entities.append((current_type, " ".join(current)))
                current = []
                current_type = None

    if current:
        entities.append((current_type, " ".join(current)))

    return entities

# ===============================
# Convert to @@ format
# ===============================
def format_sentence(tokens, entities, target_type):
    sentence = " ".join(tokens)

    for etype, text in entities:
        if etype == target_type:
            sentence = sentence.replace(text, f"@@{text}##")

    return sentence

# ===============================
# Build training examples
# ===============================
train_examples = []

for ex in dataset["train"]:
    tokens = ex["tokens"]
    entities = extract_entities(tokens, ex["ner_tags"])
    sentence = " ".join(tokens)

    train_examples.append((tokens, entities, sentence))

# ===============================
# Sample validation set (500)
# ===============================
val_data = random.sample(list(dataset["validation"]), 500)

# ===============================
# Prompt builder
# ===============================
def build_prompt(few_shot, test_sentence, entity_type):
    prompt = f"""I am an excellent linguist.
The task is to label {entity_type} entities in the given sentence.
Below are some examples.

"""

    for tokens, entities, sent in few_shot:
        labeled = format_sentence(tokens, entities, entity_type)
        prompt += f"Input: {sent}\nOutput: {labeled}\n\n"

    prompt += f"Input: {test_sentence}\nOutput:"
    return prompt

# ===============================
# GPT call
# ===============================
def call_gpt(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# ===============================
# Extract @@ entities
# ===============================
def extract_predicted(text):
    return re.findall(r"@@(.*?)##", text)

# ===============================
# Evaluation
# ===============================
TP, FP, FN = 0, 0, 0

# ===============================
# Main loop
# ===============================
k = 5

for ex in tqdm(val_data):
    tokens = ex["tokens"]
    gold_entities = extract_entities(tokens, ex["ner_tags"])
    sentence = " ".join(tokens)

    pred_all = []

    for etype in entity_types:
        few_shot = random.sample(train_examples, k)
        prompt = build_prompt(few_shot, sentence, etype)

        try:
            output = call_gpt(prompt)
        except:
            continue

        preds = extract_predicted(output)
        pred_all.extend([(etype, p) for p in preds])

    gold_set = set(gold_entities)
    pred_set = set(pred_all)

    TP += len(gold_set & pred_set)
    FP += len(pred_set - gold_set)
    FN += len(gold_set - pred_set)

# ===============================
# Metrics
# ===============================
precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print("\n===== RESULT =====")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Enter your OpenAI API key: ··········


100%|██████████| 500/500 [22:47<00:00,  2.73s/it]


===== RESULT =====
Precision: 0.5862
Recall:    0.5535
F1 Score:  0.5694


## Randon Retrieval k= 8,12

In [ ]:
k_values = [8, 12]

results_random = {}

for k in k_values:
    print(f"\nRunning RANDOM retrieval for k = {k}")

    TP, FP, FN = 0, 0, 0

    for ex in tqdm(val_data):
        tokens = ex["tokens"]
        gold_entities = extract_entities(tokens, ex["ner_tags"])
        sentence = " ".join(tokens)

        pred_all = []

        for etype in entity_types:
            # 🔥 random few-shot
            few_shot = random.sample(train_examples, k)

            prompt = build_prompt(few_shot, sentence, etype)

            try:
                output = call_gpt(prompt)
            except:
                continue

            preds = extract_predicted(output)
            pred_all.extend([(etype, p) for p in preds])

        gold_set = set(gold_entities)
        pred_set = set(pred_all)

        TP += len(gold_set & pred_set)
        FP += len(pred_set - gold_set)
        FN += len(gold_set - pred_set)

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    results_random[k] = (precision, recall, f1)

    print(f"[RANDOM] k={k} → Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")


Running RANDOM retrieval for k = 8


100%|██████████| 500/500 [25:10<00:00,  3.02s/it]


[RANDOM] k=8 → Precision=0.6012, Recall=0.6641, F1=0.6311

Running RANDOM retrieval for k = 12


100%|██████████| 500/500 [26:55<00:00,  3.23s/it]

[RANDOM] k=12 → Precision=0.6167, Recall=0.6819, F1=0.6476


## Sentence level embedding k=5

In [ ]:
!pip install transformers==4.41.2 huggingface_hub==0.23.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fol

In [ ]:
import os
os.kill(os.getpid(), 9)  # restart runtime

In [ ]:
# ===============================
# Imports
# ===============================
from datasets import load_dataset
import random
import re
from tqdm import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import getpass
from openai import OpenAI

# ===============================
# API KEY (hidden)
# ===============================
api_key = getpass.getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)

# ===============================
# Load dataset
# ===============================
dataset = load_dataset("conll2003")

# ===============================
# SimCSE model (pretrained)
# ===============================
tokenizer = AutoTokenizer.from_pretrained(
    "princeton-nlp/sup-simcse-bert-base-uncased"
)
model = AutoModel.from_pretrained(
    "princeton-nlp/sup-simcse-bert-base-uncased"
)

model.eval()

# ===============================
# Tag mapping
# ===============================
tag_map = {
    0: "O",
    1: "B-PER", 2: "I-PER",
    3: "B-ORG", 4: "I-ORG",
    5: "B-LOC", 6: "I-LOC",
    7: "B-MISC", 8: "I-MISC"
}

entity_types = ["PER", "LOC", "ORG", "MISC"]

# ===============================
# Extract entities
# ===============================
def extract_entities(tokens, tags):
    entities = []
    current = []
    current_type = None

    for token, tag in zip(tokens, tags):
        tag_name = tag_map[tag]

        if tag_name.startswith("B-"):
            if current:
                entities.append((current_type, " ".join(current)))
            current = [token]
            current_type = tag_name[2:]

        elif tag_name.startswith("I-") and current:
            current.append(token)

        else:
            if current:
                entities.append((current_type, " ".join(current)))
                current = []
                current_type = None

    if current:
        entities.append((current_type, " ".join(current)))

    return entities

# ===============================
# @@ format
# ===============================
def format_sentence(tokens, entities, target_type):
    sentence = " ".join(tokens)

    for etype, text in entities:
        if etype == target_type:
            sentence = sentence.replace(text, f"@@{text}##")

    return sentence

# ===============================
# Build train pool
# ===============================
train_examples = []
train_sentences = []

for ex in dataset["train"]:
    tokens = ex["tokens"]
    entities = extract_entities(tokens, ex["ner_tags"])
    sentence = " ".join(tokens)

    train_examples.append((tokens, entities, sentence))
    train_sentences.append(sentence)

# ===============================
# Compute embeddings (train set)
# ===============================
def embed(sentences):
    inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, return_dict=True)
        embeddings = outputs.pooler_output
    return embeddings

# batch embedding
batch_size = 64
train_embeddings = []

for i in tqdm(range(0, len(train_sentences), batch_size)):
    batch = train_sentences[i:i+batch_size]
    emb = embed(batch)
    train_embeddings.append(emb)

train_embeddings = torch.cat(train_embeddings, dim=0)

# ===============================
# Validation set (500)
# ===============================
val_data = random.sample(list(dataset["validation"]), 500)

# ===============================
# Prompt builder
# ===============================
def build_prompt(few_shot, test_sentence, entity_type):
    prompt = f"""I am an excellent linguist.
The task is to label {entity_type} entities in the given sentence.
Below are some examples.

"""

    for tokens, entities, sent in few_shot:
        labeled = format_sentence(tokens, entities, entity_type)
        prompt += f"Input: {sent}\nOutput: {labeled}\n\n"

    prompt += f"Input: {test_sentence}\nOutput:"
    return prompt

# ===============================
# GPT call
# ===============================
def call_gpt(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# ===============================
# Extract predictions
# ===============================
def extract_predicted(text):
    return re.findall(r"@@(.*?)##", text)

# ===============================
# kNN retrieval (sentence similarity)
# ===============================
def retrieve_similar(test_sentence, k=5):
    test_emb = embed([test_sentence])  # (1, dim)

    sim = F.cosine_similarity(test_emb, train_embeddings)  # (N,)
    topk = torch.topk(sim, k=k).indices.tolist()

    return [train_examples[i] for i in topk]

# ===============================
# Evaluation
# ===============================
TP, FP, FN = 0, 0, 0

# ===============================
# Main loop
# ===============================
k = 5

for ex in tqdm(val_data):
    tokens = ex["tokens"]
    gold_entities = extract_entities(tokens, ex["ner_tags"])
    sentence = " ".join(tokens)

    pred_all = []

    for etype in entity_types:
        few_shot = retrieve_similar(sentence, k)
        prompt = build_prompt(few_shot, sentence, etype)

        try:
            output = call_gpt(prompt)
        except:
            continue

        preds = extract_predicted(output)
        pred_all.extend([(etype, p) for p in preds])

    gold_set = set(gold_entities)
    pred_set = set(pred_all)

    TP += len(gold_set & pred_set)
    FP += len(pred_set - gold_set)
    FN += len(gold_set - pred_set)

# ===============================
# Metrics
# ===============================
precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print("\n===== RESULT (SimCSE) =====")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Enter your OpenAI API key: ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

100%|██████████| 500/500 [38:36<00:00,  4.63s/it]


===== RESULT (SimCSE) =====
Precision: 0.7830
Recall:    0.8187
F1 Score:  0.8004


## Sentene level embedding k= 8, 12



In [ ]:
k_values = [8, 12]

results = {}

for k in k_values:
    print(f"\nRunning for k = {k}")

    TP, FP, FN = 0, 0, 0

    for ex in tqdm(val_data):
        tokens = ex["tokens"]
        gold_entities = extract_entities(tokens, ex["ner_tags"])
        sentence = " ".join(tokens)

        pred_all = []

        for etype in entity_types:
            few_shot = retrieve_similar(sentence, k)
            prompt = build_prompt(few_shot, sentence, etype)

            try:
                output = call_gpt(prompt)
            except:
                continue

            preds = extract_predicted(output)
            pred_all.extend([(etype, p) for p in preds])

        gold_set = set(gold_entities)
        pred_set = set(pred_all)

        TP += len(gold_set & pred_set)
        FP += len(pred_set - gold_set)
        FN += len(gold_set - pred_set)

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    results[k] = (precision, recall, f1)

    print(f"k={k} → Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")


Running for k = 8


100%|██████████| 500/500 [31:21<00:00,  3.76s/it]


k=8 → Precision=0.7967, Recall=0.8543, F1=0.8245

Running for k = 12


100%|██████████| 500/500 [30:30<00:00,  3.66s/it]

k=12 → Precision=0.7992, Recall=0.8765, F1=0.8361


## Entity Embedding Retrieval

In [ ]:
# =========================
# INSTALL
# =========================
# !pip install -q datasets==2.14.6 transformers==4.41.2 huggingface_hub==0.23.4 scikit-learn tqdm openai

# =========================
# IMPORTS
# =========================
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import load_dataset
from openai import OpenAI
import getpass
import os
import re

# =========================
# 🔑 SECURE API KEY INPUT
# =========================
api_key = getpass.getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

# =========================
# LOAD DATASET
# =========================
dataset = load_dataset("conll2003")
train_data = dataset["train"]
val_data = dataset["validation"]

label_list = dataset["train"].features["ner_tags"].feature.names

# -------------------------
# GOLD ENTITY EXTRACT
# -------------------------
def extract_entities(tokens, ner_tags):
    entities = []
    current = []

    for token, tag in zip(tokens, ner_tags):
        label = label_list[tag]

        if label.startswith("B-"):
            if current:
                entities.append(" ".join(current))
            current = [token]
        elif label.startswith("I-") and current:
            current.append(token)
        else:
            if current:
                entities.append(" ".join(current))
                current = []

    if current:
        entities.append(" ".join(current))

    return set(entities)

# -------------------------
# PARSE GPT OUTPUT (@@ ##)
# -------------------------
def parse_output(output, entity_type):
    pattern = r"@@(.*?)##"
    matches = re.findall(pattern, output)
    return [m.strip() for m in matches if m.strip()]

# =========================
# LOAD NER MODEL
# =========================
model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
model.eval()

id2label = model.config.id2label

# =========================
# ENTITY + EMBEDDING
# =========================
def get_entities_and_embeddings(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    logits = outputs.logits
    hidden_states = outputs.hidden_states[-1][0]

    preds = torch.argmax(logits, dim=2)[0]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    entities = []
    current_tokens = []
    current_embs = []
    current_label = None

    for tok, pred, emb in zip(tokens, preds, hidden_states):

        if tok in ["[CLS]", "[SEP]"]:
            continue

        label = id2label[pred.item()]

        if label.startswith("B-"):
            if current_tokens:
                entities.append((current_label, current_tokens, current_embs))
            current_label = label[2:]
            current_tokens = [tok]
            current_embs = [emb]

        elif label.startswith("I-") and current_tokens:
            current_tokens.append(tok)
            current_embs.append(emb)

        else:
            if current_tokens:
                entities.append((current_label, current_tokens, current_embs))
                current_tokens = []
                current_embs = []

    if current_tokens:
        entities.append((current_label, current_tokens, current_embs))

    results = []
    for label, toks, embs in entities:
        text = tokenizer.convert_tokens_to_string(toks)
        emb = torch.stack(embs).mean(dim=0).numpy()
        results.append((label, text, emb))

    return results

# =========================
# LABEL SENTENCE (FIXED 🔥)
# =========================
def label_sentence(sentence, entity_type):
    ents = get_entities_and_embeddings(sentence)
    ents = [e[1] for e in ents if e[0] == entity_type]

    labeled = sentence

    # longer entities first (avoid substring issues)
    ents = sorted(ents, key=len, reverse=True)

    for ent in ents:
        labeled = re.sub(rf"\b{re.escape(ent)}\b", f"@@{ent}##", labeled)

    return labeled

# =========================
# BUILD DATASTORE
# =========================
datastore = {"PER": [], "ORG": [], "LOC": [], "MISC": []}

print("Building datastore...")

for ex in tqdm(train_data):
    sentence = " ".join(ex["tokens"])
    ents = get_entities_and_embeddings(sentence)

    for label, text, emb in ents:
        if label in datastore:
            datastore[label].append((emb, sentence))

# =========================
# RETRIEVAL
# =========================
def retrieve_entity_level(sentence, entity_type, k=5):
    ents = get_entities_and_embeddings(sentence)
    ents = [e for e in ents if e[0] == entity_type]

    if len(ents) == 0:
        return []

    query_embs = np.array([e[2] for e in ents])

    db = datastore[entity_type]
    if len(db) == 0:
        return []

    db_embs = np.array([x[0] for x in db])
    db_sents = [x[1] for x in db]

    sims = cosine_similarity(query_embs, db_embs)
    scores = sims.max(axis=0)

    topk_idx = scores.argsort()[-k:][::-1]
    results = [db_sents[i] for i in topk_idx]

    return list(dict.fromkeys(results))

# =========================
# PROMPT (FIXED 🔥)
# =========================
def build_prompt(examples, sentence, entity_type):
    prompt = f"I am an excellent linguist.\n"
    prompt += f"The task is to label {entity_type} entities in the given sentence.\n"
    prompt += "Below are some examples.\n\n"

    for ex in examples:
        labeled = label_sentence(ex, entity_type)
        prompt += f"Input: {ex}\nOutput: {labeled}\n\n"

    prompt += f"Input: {sentence}\nOutput:"
    return prompt

# =========================
# GPT CALL
# =========================
def run_gpt(prompt):
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=256
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("API Error:", e)
        return ""

# =========================
# EVALUATION (500)
# =========================
entity_types = ["PER", "ORG", "LOC", "MISC"]

TP, FP, FN = 0, 0, 0

print("Running evaluation...")

for ex in tqdm(val_data.select(range(500))):
    tokens = ex["tokens"]
    sentence = " ".join(tokens)

    gold_entities = extract_entities(tokens, ex["ner_tags"])
    pred_all = []

    for etype in entity_types:
        few_shot = retrieve_entity_level(sentence, etype, k=5)

        if len(few_shot) == 0:
            few_shot = [" ".join(train_data[i]["tokens"]) for i in range(5)]

        prompt = build_prompt(few_shot, sentence, etype)

        output = run_gpt(prompt)
        preds = parse_output(output, etype)

        pred_all.extend(preds)

    pred_set = set(pred_all)
    gold_set = set(gold_entities)

    TP += len(pred_set & gold_set)
    FP += len(pred_set - gold_set)
    FN += len(gold_set - pred_set)

precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print(f"\nPrecision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

Enter your OpenAI API key: ··········


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Building datastore...


100%|██████████| 14041/14041 [27:48<00:00,  8.41it/s]


Running evaluation...


100%|██████████| 500/500 [46:04<00:00,  5.53s/it]


Precision: 0.8039
Recall: 0.8389
F1: 0.8210


In [ ]:
# =========================
# EVALUATION + SELF-VERIFICATION (k = 12)
# =========================

def verify_entity(sentence, entity, entity_type):
    entity = entity.strip()

    # skip hallucinated spans (important)
    if entity not in sentence:
        return False

    prompt = f"""
The input sentence: {sentence}
Is "{entity}" in the input sentence a {entity_type} entity?
Please answer with yes or no.
"""
    res = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    ans = res.choices[0].message.content.strip().upper()
    return ans.startswith("YES")


entity_types = ["PER", "ORG", "LOC", "MISC"]
k = 12

val_subset = val_data.select(range(500))

print(f"\n===== Entity-level + Self-Verification (k = {k}) =====")

TP = FP = FN = 0

for ex in tqdm(val_subset):
    sentence = " ".join(ex["tokens"])
    gold = extract_entities(ex["tokens"], ex["ner_tags"])

    pred_all = []

    for etype in entity_types:
        examples = retrieve_entity_level(sentence, etype, k)
        if not examples:
            examples = [" ".join(train_data[i]["tokens"]) for i in range(k)]

        # Extraction
        out = run_gpt(build_prompt(examples, sentence, etype))
        preds = parse_output(out, etype)

        # Self-verification (cleaned)
        verified = []
        for ent in preds:
            ent = ent.strip()
            if verify_entity(sentence, ent, etype):
                verified.append(ent)

        pred_all.extend(verified)

    pred, gold = set(pred_all), set(gold)

    TP += len(pred & gold)
    FP += len(pred - gold)
    FN += len(gold - pred)

p = TP / (TP + FP)
r = TP / (TP + FN)
f1 = 2 * p * r / (p + r)

print(f"P={p:.4f} R={r:.4f} F1={f1:.4f}")


===== Entity-level + Self-Verification (k = 12) =====


100%|██████████| 500/500 [1:38:03<00:00, 11.77s/it]

P=0.9665 R=0.7246 F1=0.8282


In [ ]:
# =========================
# INSTALL (run once)
# =========================
!pip install -q datasets==2.14.6 transformers==4.41.2 huggingface_hub==0.23.4 scikit-learn tqdm openai

# =========================
# IMPORTS
# =========================
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import load_dataset
from openai import OpenAI
import getpass, os, re

# =========================
# API KEY
# =========================
api_key = getpass.getpass("Enter OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
client = OpenAI()

# =========================
# DATA
# =========================
dataset = load_dataset("conll2003")
train_data = dataset["train"]
val_data = dataset["validation"].select(range(50))  # ✅ only 50

label_list = dataset["train"].features["ner_tags"].feature.names

# =========================
# ENTITY EXTRACTION (GOLD)
# =========================
def extract_entities(tokens, ner_tags):
    entities, current = [], []
    for token, tag in zip(tokens, ner_tags):
        label = label_list[tag]
        if label.startswith("B-"):
            if current: entities.append(" ".join(current))
            current = [token]
        elif label.startswith("I-") and current:
            current.append(token)
        else:
            if current:
                entities.append(" ".join(current))
                current = []
    if current:
        entities.append(" ".join(current))
    return set(entities)

def parse_output(output):
    return re.findall(r"@@(.*?)##", output)

# =========================
# BERT MODEL (ENTITY EMBEDDING)
# =========================
model_name = "dslim/bert-base-NER"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)
model.eval()
id2label = model.config.id2label

def get_entities_and_embeddings(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    preds = torch.argmax(outputs.logits, dim=2)[0]
    hidden = outputs.hidden_states[-1][0]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    entities, cur_toks, cur_embs, cur_label = [], [], [], None

    for tok, pred, emb in zip(tokens, preds, hidden):
        if tok in ["[CLS]", "[SEP]"]: continue
        label = id2label[pred.item()]

        if label.startswith("B-"):
            if cur_toks:
                entities.append((cur_label, cur_toks, cur_embs))
            cur_label = label[2:]
            cur_toks, cur_embs = [tok], [emb]
        elif label.startswith("I-") and cur_toks:
            cur_toks.append(tok)
            cur_embs.append(emb)
        else:
            if cur_toks:
                entities.append((cur_label, cur_toks, cur_embs))
                cur_toks, cur_embs = [], []

    if cur_toks:
        entities.append((cur_label, cur_toks, cur_embs))

    results = []
    for label, toks, embs in entities:
        text = tokenizer.convert_tokens_to_string(toks)
        emb = torch.stack(embs).mean(dim=0).numpy()
        results.append((label, text, emb))

    return results

# =========================
# LABEL SENTENCE (@@ ##)
# =========================
def label_sentence(sentence, entity_type):
    ents = get_entities_and_embeddings(sentence)
    ents = sorted([e[1] for e in ents if e[0] == entity_type], key=len, reverse=True)
    for ent in ents:
        sentence = re.sub(rf"\b{re.escape(ent)}\b", f"@@{ent}##", sentence)
    return sentence

# =========================
# BUILD DATASTORE (entity-level)
# =========================
datastore = {"PER": [], "ORG": [], "LOC": [], "MISC": []}

for ex in tqdm(train_data):
    sentence = " ".join(ex["tokens"])
    for label, text, emb in get_entities_and_embeddings(sentence):
        if label in datastore:
            datastore[label].append((emb, sentence))

# =========================
# RETRIEVAL (entity-level kNN)
# =========================
def retrieve_entity_level(sentence, entity_type, k):
    ents = [e for e in get_entities_and_embeddings(sentence) if e[0] == entity_type]
    if not ents: return []

    query = np.array([e[2] for e in ents])
    db = datastore[entity_type]

    db_embs = np.array([x[0] for x in db])
    db_sents = [x[1] for x in db]

    sims = cosine_similarity(query, db_embs)
    scores = sims.max(axis=0)

    idx = scores.argsort()[-k:][::-1]
    return list(dict.fromkeys([db_sents[i] for i in idx]))

# =========================
# GPT EXTRACTION PROMPT
# =========================
def build_prompt(examples, sentence, entity_type):
    prompt = f"I am an excellent linguist.\nThe task is to label {entity_type} entities.\n\n"
    for ex in examples:
        prompt += f"Input: {ex}\nOutput: {label_sentence(ex, entity_type)}\n\n"
    prompt += f"Input: {sentence}\nOutput:"
    return prompt

def run_gpt(prompt):
    res = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return res.choices[0].message.content

# =========================
# 🔥 ZERO-SHOT SELF-VERIFICATION (UPGRADED)
# =========================
ENTITY_DEFINITION = {
    "PER": "persons or people's names",
    "ORG": "organizations such as companies, institutions",
    "LOC": "locations such as cities, countries, regions",
    "MISC": "miscellaneous entities like events, nationalities"
}

def verify_entity(sentence, entity, entity_type):
    if entity.lower() not in sentence.lower():
        return False

    definition = ENTITY_DEFINITION[entity_type]

    prompt = f"""
You are an excellent linguist.

The task is to verify whether the word is a {entity_type} entity.
{entity_type} entities are {definition}.

The input sentence: {sentence}
Is the word "{entity}" in the input sentence a {entity_type} entity?
Please answer with yes or no.
"""

    res = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    ans = res.choices[0].message.content.strip().lower()

    return ans.startswith("yes")

# =========================
# EVALUATION
# =========================
entity_types = ["PER", "ORG", "LOC", "MISC"]
k = 12

TP = FP = FN = 0

print("\n===== Entity-level + Zero-shot Self-Verification =====")

for ex in tqdm(val_data):
    sentence = " ".join(ex["tokens"])
    gold = extract_entities(ex["tokens"], ex["ner_tags"])

    pred_all = []

    for etype in entity_types:
        examples = retrieve_entity_level(sentence, etype, k)
        if not examples:
            examples = [" ".join(train_data[i]["tokens"]) for i in range(k)]

        # Extraction
        out = run_gpt(build_prompt(examples, sentence, etype))

        preds = parse_output(out)

        # Verification
        for ent in preds:
            if verify_entity(sentence, ent, etype):
                pred_all.append(ent)

    pred = set(pred_all)

    TP += len(pred & gold)
    FP += len(pred - gold)
    FN += len(gold - pred)

p = TP / (TP + FP + 1e-8)
r = TP / (TP + FN + 1e-8)
f1 = 2 * p * r / (p + r + 1e-8)

print(f"P={p:.4f} R={r:.4f} F1={f1:.4f}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|██████████| 14041/14041 [28:44<00:00,  8.14it/s]



===== Entity-level + Zero-shot Self-Verification =====


100%|██████████| 50/50 [09:43<00:00, 11.67s/it]

P=0.8734 R=0.5476 F1=0.6732


In [ ]:
# =========================
# VALIDATION (500) ONLY
# =========================

entity_types = ["PER", "ORG", "LOC", "MISC"]
k = 12

val_subset = dataset["validation"].select(range(500))

TP = FP = FN = 0

print("\n===== Entity-level + Zero-shot Self-Verification (500) =====")

for ex in tqdm(val_subset):
    sentence = " ".join(ex["tokens"])
    gold = extract_entities(ex["tokens"], ex["ner_tags"])

    pred_all = []

    # 🔁 4 entity types loop
    for etype in entity_types:
        examples = retrieve_entity_level(sentence, etype, k)

        if not examples:
            examples = [" ".join(train_data[i]["tokens"]) for i in range(k)]

        # 🔹 Extraction
        out = run_gpt(build_prompt(examples, sentence, etype))
        preds = parse_output(out)

        # 🔹 Zero-shot verification
        for ent in preds:
            if verify_entity(sentence, ent, etype):
                pred_all.append(ent)

    pred = set(pred_all)

    TP += len(pred & gold)
    FP += len(pred - gold)
    FN += len(gold - pred)

# =========================
# METRICS
# =========================
p = TP / (TP + FP + 1e-8)
r = TP / (TP + FN + 1e-8)
f1 = 2 * p * r / (p + r + 1e-8)

print(f"P={p:.4f} R={r:.4f} F1={f1:.4f}")


===== Entity-level + Zero-shot Self-Verification (500) =====


100%|██████████| 500/500 [1:28:04<00:00, 10.57s/it]

P=0.9669 R=0.7669 F1=0.8553
